# Sprint 5 - Data Cleaning & Data Preprocessing

## Notebook 1 - Data Preprocessing Basics

In this notebook I learn the basics of preprocessing before I clean anything. I cover what preprocessing is, why it is needed, raw vs clean data, data quality, data leakage, train/validation/test data and the preprocessing workflow.

**Dataset:** Telco Customer Churn (7043 rows, 21 columns), same dataset as Sprint 4.

**Rule for this sprint:** `df` (the raw data) is never changed. I do every experiment on a copy.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")   # raw data, never modified
print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Problems Identified in Sprint 4

Before I clean the data, I write down the problems I found in Sprint 4. I solve each one in the notebook shown in the last column.

| # | Problem | Why it exists | Planned fix | Notebook |
|---|---|---|---|---|
| 1 | `TotalCharges` is stored as text | 11 cells have a blank space | convert with `to_numeric` | 02 |
| 2 | 11 hidden missing values in `TotalCharges` | those customers have tenure 0, so no bill yet | fill by business rule (0) or median | 03 |
| 3 | No duplicate rows | one row per customer | keep a duplicate check | 04 |
| 4 | "No internet service" / "No phone service" repeat information | same fact stored in two columns | merge into "No" or encode carefully | 05, 07 |
| 5 | `customerID` is useless for prediction | it is only a key | drop it | 10 |
| 6 | `TotalCharges` is skewed and correlated with `tenure` | it is roughly tenure x MonthlyCharges | transform, check in feature selection | 09, 10 |
| 7 | Churn is imbalanced (about 26.5% yes) | most customers stay | class weights / SMOTE on train only | 11 |

In [4]:
blank = df["TotalCharges"].astype(str).str.strip().eq("")
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Blank TotalCharges:", blank.sum(), "| their tenure:", df.loc[blank, "tenure"].unique())
print("Missing values found by isna():", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum(), "| duplicate customerID:", df["customerID"].duplicated().sum())
print("'No internet service' in OnlineSecurity:", (df["OnlineSecurity"] == "No internet service").sum())
print("'No phone service' in MultipleLines:", (df["MultipleLines"] == "No phone service").sum())

total = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Skew of TotalCharges:", round(total.skew(), 2))
print("Correlation tenure vs TotalCharges:", round(df["tenure"].corr(total), 2))
print("Churn rate (%):", round((df["Churn"] == "Yes").mean() * 100, 1))

TotalCharges dtype: str
Blank TotalCharges: 11 | their tenure: [0]
Missing values found by isna(): 0
Duplicate rows: 0 | duplicate customerID: 0
'No internet service' in OnlineSecurity: 1526
'No phone service' in MultipleLines: 682
Skew of TotalCharges: 0.96
Correlation tenure vs TotalCharges: 0.83
Churn rate (%): 26.5


**What the code does:** It checks the problems of the table above with numbers.

**Why this method was used:** Each problem should have a proof, not only my memory.

**Output meaning:** `isna()` finds nothing, but 11 blank strings are hidden in `TotalCharges` and all of them have tenure 0.

**Effect on ML:** These problems are what the next notebooks fix. If I skip them, the model cannot train or the result is misleading.

## 2. What is Data Preprocessing?

**Concept:** Data preprocessing means converting raw data into a clean format that a machine learning model can learn from. Raw data is collected for business work, not for ML, so it has blanks, wrong types and text values.

**Examples:**
- Simple: a column "Age" has 25, 31 and "thirty", so the text must be fixed first.
- Real-world: one nurse writes "M" and another writes "Male", so the model sees two groups.
- Business: `TotalCharges` is text, so I cannot calculate the average bill.
- AI/ML: logistic regression and SVM accept only numbers, so text columns must be encoded.

**Implementation:** I give the raw data directly to a model and see what happens.

In [5]:
from sklearn.linear_model import LogisticRegression

X_raw = df.drop(columns=["Churn"])
y_raw = (df["Churn"] == "Yes").astype(int)

try:
    LogisticRegression(max_iter=1000).fit(X_raw, y_raw)
    print("trained")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:120])

ValueError : could not convert string to float: '7590-VHVEG'


**What the code does:** It trains logistic regression on the raw features.

**Why this method was used:** The easiest way to show why preprocessing is needed is to show the error without it.

**Before:** features contain text like `customerID` and `Contract`.

**Result:** The model gives an error because it cannot convert text to a number.

**Effect on ML:** A model has input rules (numbers only, no blanks), and preprocessing makes the data follow them.

## 3. Why is Data Preprocessing Required?

**Concept:** A model cannot know which values are wrong, so if I give bad data it learns bad patterns ("garbage in, garbage out"). I need preprocessing because models need numeric input, bad data gives bad learning, features with big values can dominate small ones, and a wrong split gives a fake score.

**Examples:**
- Simple: age (20 to 60) and salary (20,000 to 200,000), so without scaling KNN uses mostly salary.
- Real-world: a bank has duplicate loan records, so the model becomes too sure about those borrowers.
- Business: dropping the 11 blank `TotalCharges` rows silently removes all brand new customers.
- AI/ML: a churn model needs numeric input, a fair target and no future information.

**Implementation:** I check how many columns of the dataset are ready for a model.

In [6]:
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
text_cols = [c for c in df.columns if c not in numeric_cols]

print("Numeric columns:", numeric_cols)
print("Text columns   :", len(text_cols), "of", df.shape[1])
print("\nNumeric looking columns stored as text:", [c for c in text_cols if pd.to_numeric(df[c], errors="coerce").notna().sum() > 0.9 * len(df)])
print("Columns needing encoding:", len(text_cols) - 1, "(customerID is dropped, TotalCharges is converted)")

Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Text columns   : 18 of 21

Numeric looking columns stored as text: ['TotalCharges']
Columns needing encoding: 17 (customerID is dropped, TotalCharges is converted)


**What the code does:** It counts numeric and text columns and finds numbers hidden as text.

**Why this method was used:** It shows quickly how much work the dataset needs.

**Before:** only `tenure`, `MonthlyCharges` and `SeniorCitizen` are numeric.

**Result:** Most columns are text, and `TotalCharges` is a number stored as text.

**Effect on ML:** I cannot apply one operation to everything. Numeric and categorical columns need different steps, so later I use a `ColumnTransformer`.

## 4. Raw Data vs Clean Data

**Concept:** Raw data is the data as it was collected, with all its problems. Clean data is a copy after the problems are corrected with rules I can explain. Raw data is never overwritten.

**Examples:**
- Simple: "  Yes", "yes" and "Y" become "Yes".
- Real-world: the date "03/04/2024" becomes one clear date format.
- Business: a blank `TotalCharges` of a new customer becomes 0.
- AI/ML: text and blanks in a numeric column become a float column that can be scaled.

**Implementation:** I compare raw and clean `TotalCharges` (only a preview, full work is in Notebooks 2 and 3).

In [7]:
demo = df[["tenure", "MonthlyCharges", "TotalCharges"]].copy()

demo["TotalCharges_clean"] = pd.to_numeric(demo["TotalCharges"], errors="coerce")
new_customer = demo["tenure"] == 0
demo.loc[new_customer, "TotalCharges_clean"] = demo.loc[new_customer, "TotalCharges_clean"].fillna(0)

print("RAW   dtype:", demo["TotalCharges"].dtype, "| blank strings:", demo["TotalCharges"].astype(str).str.strip().eq("").sum())
print("CLEAN dtype:", demo["TotalCharges_clean"].dtype, "| missing:", demo["TotalCharges_clean"].isna().sum())
print("CLEAN mean :", round(demo["TotalCharges_clean"].mean(), 2))
print()
print(demo[new_customer].head())

RAW   dtype: str | blank strings: 11
CLEAN dtype: float64 | missing: 0
CLEAN mean : 2279.73

      tenure  MonthlyCharges TotalCharges  TotalCharges_clean
488        0           52.55                              0.0
753        0           20.25                              0.0
936        0           80.85                              0.0
1082       0           25.75                              0.0
1340       0           56.05                              0.0


**What the code does:** It converts `TotalCharges` to numbers and fills the blanks of tenure 0 customers with 0.

**Why this method was used:** `errors="coerce"` turns only the bad values into missing, so valid values are not lost. I used 0 because a customer with tenure 0 has not paid anything yet.

**Before:** text column, 11 blank strings.

**After:** float column, no missing values, mean can be calculated.

**Why appropriate:** Dropping the 11 rows loses new customers, and the mean would give them a bill that does not exist.

**Effect on ML:** The column can now be scaled. I compare 0 with median imputation in Notebook 3.

## 5. Data Quality

**Concept:** Data quality tells how good the data is for its purpose. It is not one number, it has dimensions, and a dataset can be good in one and bad in another.

| Dimension | Meaning | Example |
|---|---|---|
| Completeness | Are all values present? | an email column with many empty cells |
| Consistency | Is the same fact written the same way? | "Male", "M" and "male" |
| Accuracy | Do values match the real world? | age 250, or a total that does not match its parts |
| Validity | Do values follow the allowed range, format or set? | negative salary |
| Integrity | Do the relations between values hold? | two customers with the same ID |

**Examples (business and AI/ML):**
- Completeness: blank `TotalCharges`, so rows are dropped or need imputation.
- Consistency: "No" and "No internet service" mean the same, so one group is split into two.
- Accuracy: a bill that does not match tenure x price, so the model learns the error.
- Validity: a negative monthly charge distorts scaling.
- Integrity: a duplicate `customerID` can appear in both train and test and inflate the score.

**Implementation:** I check each dimension on the Telco data.

In [8]:
# Completeness
as_missing = df.replace(r"^\s*$", np.nan, regex=True)
print("COMPLETENESS")
print("  missing by isna()          :", int(df.isna().sum().sum()))
print("  missing incl. blank strings:", int(as_missing.isna().sum().sum()))

# Consistency
net_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
net_ok = all(((df[c] == "No internet service") == (df["InternetService"] == "No")).all() for c in net_cols)
phone_ok = ((df["MultipleLines"] == "No phone service") == (df["PhoneService"] == "No")).all()
print("\nCONSISTENCY")
print("  internet labels agree with InternetService:", net_ok)
print("  phone label agrees with PhoneService      :", bool(phone_ok))

# Accuracy (plausibility check): TotalCharges should be close to tenure x MonthlyCharges
chk = df[df["tenure"] > 0].copy()
chk["TotalCharges"] = pd.to_numeric(chk["TotalCharges"], errors="coerce")
ratio = chk["TotalCharges"] / (chk["tenure"] * chk["MonthlyCharges"])
print("\nACCURACY")
print("  share of customers with ratio between 0.5 and 1.5:", round(ratio.between(0.5, 1.5).mean() * 100, 1), "%")

# Validity
rules = {
    "tenure >= 0": df["tenure"] >= 0,
    "MonthlyCharges > 0": df["MonthlyCharges"] > 0,
    "SeniorCitizen in {0,1}": df["SeniorCitizen"].isin([0, 1]),
    "gender in {Male, Female}": df["gender"].isin(["Male", "Female"]),
    "Churn in {Yes, No}": df["Churn"].isin(["Yes", "No"]),
    "Contract in 3 types": df["Contract"].isin(["Month-to-month", "One year", "Two year"]),
}
print("\nVALIDITY (rule violations)")
for name, ok in rules.items():
    print("  ", name, "->", int((~ok).sum()))
print("   TotalCharges not readable as number ->", int(pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum()))

# Integrity
blank = df["TotalCharges"].astype(str).str.strip().eq("")
print("\nINTEGRITY")
print("  customerID unique:", df["customerID"].is_unique)
print("  duplicate rows   :", int(df.duplicated().sum()))
print("  blank TotalCharges exactly when tenure == 0:", bool((blank == (df["tenure"] == 0)).all()))

COMPLETENESS
  missing by isna()          : 0
  missing incl. blank strings: 11

CONSISTENCY
  internet labels agree with InternetService: True
  phone label agrees with PhoneService      : True

ACCURACY
  share of customers with ratio between 0.5 and 1.5: 100.0 %

VALIDITY (rule violations)
   tenure >= 0 -> 0
   MonthlyCharges > 0 -> 0
   SeniorCitizen in {0,1} -> 0
   gender in {Male, Female} -> 0
   Churn in {Yes, No} -> 0
   Contract in 3 types -> 0
   TotalCharges not readable as number -> 11

INTEGRITY
  customerID unique: True
  duplicate rows   : 0
  blank TotalCharges exactly when tenure == 0: True


**What the code does:** It runs one small check for each quality dimension.

**Why this method was used:** Every dimension needs a different check, and each check gives a number I can use in the report.

**Output meaning:**
- Completeness: `isna()` finds 0 but the blank strings show 11 hidden missing values.
- Consistency: the extra labels appear exactly where the main column says "No", so there is no contradiction, but the same fact is stored twice.
- Accuracy: most ratios are close to 1, so `TotalCharges` is believable (it is not exactly 1 because prices can change).
- Validity: all rules pass, only `TotalCharges` is not readable as a number.
- Integrity: the key is unique, no duplicates, and blanks match tenure 0.

**Effect on ML:** The data is mostly good. The real issues are the type and the 11 hidden blanks of `TotalCharges`, so I only apply the techniques that the data needs.

## 6. Data Leakage

**Concept:** Data leakage means the model learns from information it will not have when it makes a real prediction. The score looks very good in development and fails in production.
- **Target leakage:** a feature contains the answer or is created after it.
- **Train-test contamination:** information from the test data enters training, for example fitting a scaler on the full data before splitting.

**Examples:**
- Simple: predicting if a student passes using the column "final grade".
- Real-world: predicting readmission using a note written after the readmission.
- Business: predicting churn with "cancellation survey completed", which only churned customers fill.
- AI/ML: scaling the whole dataset first and splitting after, so test rows affect the mean.

**Implementation:** I train a model with honest features, then add an invented column made from the target, and compare. (More types are in Notebooks 12 and 13.)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

y = (df["Churn"] == "Yes").astype(int)

def score(features):
    X_tr, X_te, y_tr, y_te = train_test_split(features, y, test_size=0.2, stratify=y, random_state=42)
    model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
    return accuracy_score(y_te, model.predict(X_te))

honest = df[["tenure", "MonthlyCharges"]].copy()
leaky = honest.copy()
leaky["cancellation_survey_done"] = y     # invented, only churned customers would have it

print("Always predict 'No churn':", round(1 - y.mean(), 3))
print("Honest features          :", round(score(honest), 3))
print("With leaking feature     :", round(score(leaky), 3))

Always predict 'No churn': 0.735
Honest features          : 0.783
With leaking feature     : 1.0


**What the code does:** It trains the same tree twice on the same split, once without and once with a leaking column.

**Why this method was used:** Only the leaking feature changes, so the score difference is caused by leakage.

**Output meaning:** The honest model is only a little better than the baseline, and the leaking model is almost perfect.

**Effect on ML:** A perfect score is a warning, not a success. For every feature I ask "will I know this value when I make the prediction?", and I fit all preprocessing on training data only.

## 7. Training, Validation and Test Data

**Concept:** A model must be tested on data it has never seen, so the data is split into three parts.
- **Training data:** the model learns from it, and all preprocessing (medians, scaler, encoder) is fitted on it.
- **Validation data:** used to compare models and tune settings.
- **Test data:** the final exam, used only once.

**Examples:**
- Simple: 100 rows split into 60 / 20 / 20.
- Real-world: lessons (train), mock test (validation), official test (test).
- Business: a churn model is trained on old customers and reported on an untouched group.
- AI/ML: a scaler is fitted on train and only `transform` is used on validation and test.

**Implementation:** I split the data 60/20/20 with `stratify` (so the churn rate stays the same) and `random_state` (so the split can be repeated).

In [10]:
X = df.drop(columns=["Churn"])
y = (df["Churn"] == "Yes").astype(int)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

summary = pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "churn_rate_%": [round(s.mean() * 100, 1) for s in (y_train, y_val, y_test)],
}, index=["train", "validation", "test"])
print(summary)

ids = [set(p["customerID"]) for p in (X_train, X_val, X_test)]
print("Customers in more than one part:", len(ids[0] & ids[1]) + len(ids[0] & ids[2]) + len(ids[1] & ids[2]))

            rows  churn_rate_%
train       4225          26.5
validation  1409          26.5
test        1409          26.5
Customers in more than one part: 0


**What the code does:** It splits off 20% for test, then takes 25% of the rest for validation (20% of the total), and prints sizes and churn rates.

**Why this method was used:** Two `train_test_split` calls are the simplest way to get three parts, and `stratify` keeps the class ratio equal.

**Result:** 60% / 20% / 20% with almost the same churn rate, and no customer in two parts.

**Effect on ML:** Scores of validation and test are now comparable. Next I show why the preprocessing must learn only from the train part.

In [11]:
X_train, X_val, X_test = X_train.copy(), X_val.copy(), X_test.copy()
for part in (X_train, X_val, X_test):
    part["TotalCharges"] = pd.to_numeric(part["TotalCharges"], errors="coerce")   # a fixed rule, not learned from data

full_median = pd.to_numeric(df["TotalCharges"], errors="coerce").median()
train_median = X_train["TotalCharges"].median()

print("WRONG  : median of full data  :", round(full_median, 2))
print("CORRECT: median of train only :", round(train_median, 2))

X_val["TotalCharges"] = X_val["TotalCharges"].fillna(train_median)
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(train_median)
print("Missing left in val / test:", X_val["TotalCharges"].isna().sum(), "/", X_test["TotalCharges"].isna().sum())

WRONG  : median of full data  : 1397.48
CORRECT: median of train only : 1385.3
Missing left in val / test: 0 / 0


**What the code does:** It calculates the median from the full data (wrong) and from the train data (correct), then fills validation and test with the train median.

**Why this method was used:** A type conversion is a fixed rule, but a median is learned from data, so it must come from train only.

**Output meaning:** The two medians are different, so the test rows already influenced the result when I used the full data.

**Effect on ML:** The difference is small here, but for scaling, target encoding, SMOTE or feature selection it can change the score. The rule is `fit` on train and `transform` on the rest, and this is why I use pipelines in Notebook 14.

## 8. Preprocessing Pipeline

**Concept:** A preprocessing pipeline is the ordered list of steps that turns raw data into model-ready data. The order matters (I cannot scale text). Steps are of two kinds:
- **Fixed rules (before the split):** fix data types, validate, remove exact duplicates, drop identifiers. They do not learn from data.
- **Learned steps (after the split, fit on train only):** imputation values, outlier limits, encoders, scalers, transformations, feature selection, SMOTE.

SMOTE or undersampling is used on the training data only, because validation and test must keep the real class ratio.

```
Load -> Inspect -> Data types -> Missing values -> Duplicates -> Validation
   ---------------- SPLIT (train / validation / test) ----------------
-> Outliers -> Encoding -> Scaling -> Transformation -> Feature selection
-> Class imbalance (train only) -> Pipeline -> Validate final dataset
```

**Examples:**
- Simple: a recipe, wash then cut then cook.
- Real-world: a factory line where each station uses the output of the previous one.
- Business: the monthly churn scoring must run the same steps in the same order every time.
- AI/ML: a `Pipeline` with imputer, scaler and encoder, and a `ColumnTransformer` for column groups.

**Implementation:** A small preview. The full pipeline is in Notebook 14 and the final one in Notebook 16.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X_train.columns if c not in num_cols + ["customerID"]]

numeric_branch = Pipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale", StandardScaler())])
categorical_branch = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                               ("encode", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([("num", numeric_branch, num_cols),
                                ("cat", categorical_branch, cat_cols)])

preprocess.fit(X_train)                      # learn from train only

for name, part in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    print(name, ":", part.shape, "->", preprocess.transform(part).shape)

train : (4225, 20) -> (4225, 46)
validation : (1409, 20) -> (1409, 46)
test : (1409, 20) -> (1409, 46)


**What the code does:** Numeric columns get median imputation and then scaling, categorical columns get most-frequent imputation and then one-hot encoding. The transformer is fitted on train and then applied to all three parts. `customerID` is dropped.

**Why this method was used:** A pipeline keeps the steps in a fixed order and fits only on train, so leakage is prevented by the structure. `handle_unknown="ignore"` avoids errors when a new category appears.

**Before:** 20 feature columns, text and numbers, with blanks.

**After:** only numbers, no missing values, more columns (one per category), and the same columns in all three parts.

**Effect on ML:** The data now has the format a model needs. This is only a preview, because outliers, skew, feature selection and class balancing are added later.

## 9. Documentation of Preprocessing Decisions

**Concept:** Every preprocessing step changes the data, so I must be able to explain it. For each decision I write seven items: Problem, Analysis, Technique selected, Reason, Implementation, Result, Impact.

**Example (TotalCharges):**

| Item | Documentation |
|---|---|
| Problem | `TotalCharges` is text and has 11 blank values |
| Analysis | The blanks are customers with tenure 0 who have no bill yet |
| Technique selected | `to_numeric(errors="coerce")` and then 0 for tenure 0 |
| Reason | 0 matches the real meaning, while mean or median gives a bill that does not exist, and dropping rows loses new customers |
| Implementation | Section 4 (preview), full version in Notebooks 2 and 3 |
| Result | text column became a float column with no missing values |
| Impact | the column can be scaled and the model still learns from new customers |

**Implementation:** I save the problems of Section 1 as an issue log. It is part of the deliverables, and I update the status in later notebooks.

In [13]:
issue_log = pd.DataFrame([
    [1, "TotalCharges", "stored as text", "11 blank strings", "to_numeric(errors='coerce')", "Notebook 2", "pending"],
    [2, "TotalCharges", "11 hidden missing values", "tenure 0, no bill yet", "business rule 0 vs median", "Notebook 3", "pending"],
    [3, "all rows", "duplicate check", "none found", "keep duplicated() check", "Notebook 4", "verified"],
    [4, "internet add-ons, MultipleLines", "redundant 'No ... service' labels", "same fact stored twice", "merge into 'No' or encode carefully", "Notebooks 5, 7", "pending"],
    [5, "customerID", "no predictive meaning", "it is only a key", "drop", "Notebook 10", "pending"],
    [6, "TotalCharges", "skewed and correlated with tenure", "about tenure x MonthlyCharges", "transform, feature selection check", "Notebooks 9, 10", "pending"],
    [7, "Churn", "class imbalance", "most customers stay", "class weights / SMOTE on train only", "Notebook 11", "pending"],
], columns=["id", "column", "problem", "analysis", "technique", "notebook", "status"])

issue_log.to_csv("data_quality_issue_log.csv", index=False)
issue_log

,id,column,problem,analysis,technique,notebook,status
0,1,TotalCharges,stored as text,11 blank strings,to_numeric(errors='coerce'),Notebook 2,pending
1,2,TotalCharges,11 hidden missing values,"tenure 0, no bill yet",business rule 0 vs median,Notebook 3,pending
2,3,all rows,duplicate check,none found,keep duplicated() check,Notebook 4,verified
3,4,"internet add-ons, MultipleLines",redundant 'No ... service' labels,same fact stored twice,merge into 'No' or encode carefully,"Notebooks 5, 7",pending
4,5,customerID,no predictive meaning,it is only a key,drop,Notebook 10,pending
5,6,TotalCharges,skewed and correlated with tenure,about tenure x MonthlyCharges,"transform, feature selection check","Notebooks 9, 10",pending
6,7,Churn,class imbalance,most customers stay,class weights / SMOTE on train only,Notebook 11,pending


**What the code does:** It saves the problems as `data_quality_issue_log.csv`.

**Why this method was used:** A file can be updated in every notebook and it connects problem, reason and technique.

**Effect on ML:** It is my checklist for Notebook 16, and every row needs a result by then.

# Summary

| Topic | Key idea |
|---|---|
| Preprocessing | converts raw data to a format the model accepts |
| Why required | models need numbers, clean data and fair features |
| Raw vs clean | raw is kept, clean is a documented copy |
| Data quality | completeness, consistency, accuracy, validity, integrity |
| Data leakage | the model must not see future or test information |
| Train / validation / test | learn, tune and test on different data |
| Pipeline | fixed rules before the split, learned steps after it, fit on train only |

**Key findings on Telco data:**
- `isna()` says the data is complete, but `TotalCharges` hides 11 blanks (tenure 0 customers).
- The real errors are only the type and the blanks of `TotalCharges`. The other issues are redundancy, skew and imbalance.
- Anything learned from data (median, scaler, encoder, SMOTE) is fitted on train only.

**What I learned:** Every preprocessing step needs a reason I can explain. I also learned that `isna()` alone is not enough to check missing values.